In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel

def process(df):
    df['label'] = (df['num DAC_RANK=1'] > 0).astype(int)
    df = df.drop(columns=['idxw', 'num DAC_RANK=1', 'num DAC_RANK>=2'])
    return df

ntrain = 1
tot = 8
train = pd.concat([ process(pd.read_csv(f'mv3_{i}.csv', index_col=None)) for i in range(ntrain) ])
test = pd.concat([ process(pd.read_csv(f'mv3_{i}.csv', index_col=None)) for i in range(ntrain, tot)])

X_train = train.drop(columns=['label'])
y_train = train['label']

X_test = test.drop(columns=['label'])
y_test = test['label']


In [9]:

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Creazione e allenamento del modello Random Forest
model = RandomForestClassifier(random_state=42)
model.fit(X_train_scaled, y_train)

# Selezione delle caratteristiche più importanti utilizzando il modello Random Forest
selector = SelectFromModel(model, threshold="mean", max_features=5)  # Seleziona le prime 5 caratteristiche
X_train_selected = selector.transform(X_train_scaled)
X_test_selected = selector.transform(X_test_scaled)

# Creazione e allenamento del modello sui dati con le caratteristiche selezionate
model_selected = RandomForestClassifier(random_state=42)
model_selected.fit(X_train_selected, y_train)

# Predizione dei risultati sui giorni successivi
y_pred = model_selected.predict(X_test_selected)

# Valutazione del modello
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

# Visualizzazione delle caratteristiche selezionate
selected_features = X_train.columns[selector.get_support()]
print("Caratteristiche selezionate:", selected_features)

Accuracy: 0.8251333728512151
Classification Report:
               precision    recall  f1-score   support

           0       0.38      0.64      0.48       211
           1       0.94      0.85      0.89      1476

    accuracy                           0.83      1687
   macro avg       0.66      0.75      0.69      1687
weighted avg       0.87      0.83      0.84      1687

Caratteristiche selezionate: Index(['qr', 'nx', 'qr_p1', 'q_p1', 'nx_p1'], dtype='object')
